# K-Means Clustering Analysis on Political Embeddings

This notebook loads precomputed $d=768$ dense political embeddings from `data/out/political_embeddings.parquet`, fits K-Means clustering with 3 components (representing Left, Center, and Right ideological groupings), profiles cluster characteristics, projects onto PCA dimensions, and generates word clouds of dominant vocabulary.

### Workflow:
1. **Load Embeddings**: Read all ~112,000 political embeddings from `data/out/political_embeddings.parquet`.
2. **K-Means Clustering**: Fit MiniBatchKMeans ($K=3$ components) across the normalized embedding space.
3. **Cluster Characterization**: Profile clusters by ideological slant distributions and top creator channels.
4. **Centroid Exemplars**: Discover the nearest-neighbor videos to each cluster centroid.
5. **PCA Projections**: Project onto PCA components to compare unsupervised clusters against ground-truth slant.
6. **Word Clouds**: Generate and display word clouds representing the dominant vocabulary for each of the 3 clusters.
7. **Export**: Save clustered assignments to `data/out/political_clusters.parquet`.


In [ ]:
import sys
from pathlib import Path

# Ensure src directory is in Python path
ROOT = Path("../")
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.decomposition import PCA
from wordcloud import WordCloud, STOPWORDS

from config import OUT_DIR

# Plotting aesthetics
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120


## 1. Load Precomputed Political Embeddings
Load the dense embeddings generated by `nbs/inference.ipynb`.


In [ ]:
embeddings_path = OUT_DIR / "political_embeddings.parquet"
print(f"Reading embeddings from: {embeddings_path}")
df_embeddings = pl.read_parquet(embeddings_path)

print("Extracting embedding matrix...")
embeddings = np.array(df_embeddings["embedding"].to_list(), dtype=np.float32)
print(f"Loaded {len(embeddings):,} embeddings. Matrix shape: {embeddings.shape}")
df_embeddings.select(["id", "channel", "title", "slant", "political_saliency"]).head(5)


## 2. Perform K-Means Clustering (K=3 Components)
Fit K-Means clustering on the normalized embedding space using `MiniBatchKMeans` with $K=3$ components.


In [ ]:
# Set number of clusters
N_CLUSTERS = 3  # 3 ideological components: Left, Center, Right
SEED = 42

print(f"Fitting MiniBatchKMeans with K={N_CLUSTERS} on {len(embeddings):,} embeddings...")
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=2048,
    random_state=SEED,
    n_init="auto"
)

cluster_labels = kmeans.fit_predict(embeddings)

# Create analysis pandas dataframe
df_analysis = df_embeddings.select(["id", "channel", "title", "slant", "political_saliency"]).to_pandas()
df_analysis["cluster"] = cluster_labels
print(f"Clustering complete. Assigned {len(cluster_labels):,} videos across {N_CLUSTERS} clusters.")

## 3. Cluster Characterization & Ideological Distribution
Inspect the ideological slant profile of each cluster (mean, median, standard deviation, and video count).

In [ ]:
# Compute summary statistics per cluster
cluster_summary = (
    df_analysis.groupby("cluster")
    .agg(
        count=("id", "count"),
        mean_slant=("slant", "mean"),
        median_slant=("slant", "median"),
        std_slant=("slant", "std"),
        mean_saliency=("political_saliency", "mean")
    )
    .reset_index()
    .sort_values("mean_slant")  # Sort from Left to Right
)

cluster_summary["share"] = (cluster_summary["count"] / len(df_analysis)).apply(lambda x: f"{x:.1%}")
print("=== Cluster Summary (Sorted by Mean Slant) ===")
display(cluster_summary.style.format({
    "mean_slant": "{:+0.3f}",
    "median_slant": "{:+0.3f}",
    "std_slant": "{:.3f}",
    "mean_saliency": "{:.3f}"
}))

# Plot Ideological Slant Distribution by Cluster
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=df_analysis,
    x="cluster",
    y="slant",
    order=cluster_summary["cluster"].tolist(),
    palette="coolwarm",
    ax=ax
)
ax.axhline(0, color="grey", linestyle="--", alpha=0.5)
ax.set_title("Distribution of Ideological Slant Across K-Means Clusters", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Cluster (Ordered from Most Left-Leaning to Most Right-Leaning)", fontsize=12)
ax.set_ylabel("Ideological Slant (-1.0: Left, +1.0: Right)", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Top Channels & Centroid Exemplar Videos per Cluster
Discover the institutional alignment and most representative video closest to each cluster's centroid.

In [ ]:
centroids = kmeans.cluster_centers_

# Normalize centroids to unit sphere for cosine distance
centroids_norm = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)

for _, row in cluster_summary.iterrows():
    k = int(row["cluster"])
    cluster_mask = df_analysis["cluster"] == k
    cluster_videos = df_analysis[cluster_mask]
    cluster_embs = embeddings[cluster_mask]
    
    # Calculate cosine distance to centroid
    similarities = np.dot(cluster_embs, centroids_norm[k])
    medoid_idx = np.argmax(similarities)
    medoid_video = cluster_videos.iloc[medoid_idx]
    
    # Top 4 channels
    top_ch = cluster_videos["channel"].value_counts().head(4).to_dict()
    top_ch_str = ", ".join([f"{ch} ({cnt:,})" for ch, cnt in top_ch.items()])
    
    print(f"\n" + "="*80)
    print(f"🔹 Cluster {k} | N={len(cluster_videos):,} ({row['share']}) | Mean Slant: {row['mean_slant']:+.3f}")
    print(f"   Top Channels: {top_ch_str}")
    print(f"   Centroid Exemplar Video:")
    print(f"     • Channel: {medoid_video['channel']}")
    print(f"     • Title:   {medoid_video['title']}")
    print(f"     • Slant:   {medoid_video['slant']:+.3f} | Saliency: {medoid_video['political_saliency']:.2f}")

## 5. 2D PCA Projections
Project a sample of 10,000 political videos onto a 2D PCA plane to compare K-Means cluster boundaries against ideological slant.

In [ ]:
PLOT_SAMPLE_SIZE = min(10000, len(df_analysis))
sample_indices = np.random.RandomState(42).choice(len(df_analysis), size=PLOT_SAMPLE_SIZE, replace=False)

plot_sample_df = df_analysis.iloc[sample_indices].copy()
sample_embs = embeddings[sample_indices]

print(f"Fitting PCA on {PLOT_SAMPLE_SIZE:,} samples...")
pca = PCA(n_components=3, random_state=42)
coords = pca.fit_transform(sample_embs)

plot_sample_df["pca_1"] = coords[:, 0]
plot_sample_df["pca_2"] = coords[:, 1]
plot_sample_df["pca_3"] = coords[:, 2]
var_exp = pca.explained_variance_ratio_

# Side-by-side plot: K-Means Clusters vs. Continuous Slant
fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)

# 1. Colored by K-Means Cluster
sns.scatterplot(
    data=plot_sample_df,
    x="pca_1",
    y="pca_2",
    hue="cluster",
    palette="tab10",
    alpha=0.6,
    s=35,
    ax=axes[0],
    legend="full"
)
axes[0].set_title(f"K-Means Clusters (K={N_CLUSTERS})", fontsize=13, fontweight="bold")
axes[0].set_xlabel(f"PC1 ({var_exp[0]:.1%} variance)")
axes[0].set_ylabel(f"PC2 ({var_exp[1]:.1%} variance)")
axes[0].axhline(0, color="grey", linestyle="--", alpha=0.3)
axes[0].axvline(0, color="grey", linestyle="--", alpha=0.3)

# 2. Colored by Ground-Truth Slant
sc = axes[1].scatter(
    plot_sample_df["pca_1"],
    plot_sample_df["pca_2"],
    c=plot_sample_df["slant"],
    cmap="coolwarm",
    alpha=0.6,
    s=35
)
cb = plt.colorbar(sc, ax=axes[1], shrink=0.85, pad=0.03)
cb.set_label("Ideological Slant (-1.0: Left, +1.0: Right)", fontsize=11)
axes[1].set_title("Ground-Truth Ideological Slant", fontsize=13, fontweight="bold")
axes[1].set_xlabel(f"PC1 ({var_exp[0]:.1%} variance)")
axes[1].axhline(0, color="grey", linestyle="--", alpha=0.3)
axes[1].axvline(0, color="grey", linestyle="--", alpha=0.3)

plt.suptitle(f"Latent Space Representation of Political Videos (N={PLOT_SAMPLE_SIZE:,})", fontsize=15, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

## 6. Word Clouds for Each Cluster
Visualize the most salient vocabulary, topics, and terminology characterizing each of the 3 political clusters.
* **Cluster Left**: Progressive, institutional, and Democratic political discourse
* **Cluster Center**: International affairs, global news, historical commentary, and bipartisan topics
* **Cluster Right**: Conservative commentary, GOP politics, and right-leaning themes


In [ ]:
# Define stopwords to filter out generic video/broadcast metadata terms
custom_stopwords = set(STOPWORDS).union({
    "video", "full", "episode", "live", "news", "watch", "part", "official",
    "new", "highlights", "show", "s", "say", "says", "one", "will", "tv"
})

# Order clusters from Left to Right by mean slant
ordered_clusters = cluster_summary.sort_values("mean_slant")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

colormaps = ["Blues_r", "viridis", "Reds_r"]
cluster_titles = ["Left-Leaning Cluster", "Center / Global Cluster", "Right-Leaning Cluster"]

for i, (_, row) in enumerate(ordered_clusters.iterrows()):
    k = int(row["cluster"])
    cluster_titles_text = " ".join(
        df_analysis[df_analysis["cluster"] == k]["title"]
        .dropna()
        .sample(min(8000, len(df_analysis[df_analysis["cluster"] == k])), random_state=SEED)
    )

    wc = WordCloud(
        width=900,
        height=550,
        background_color="white",
        colormap=colormaps[i],
        stopwords=custom_stopwords,
        max_words=80,
        collocations=False,
        random_state=SEED
    ).generate(cluster_titles_text)

    axes[i].imshow(wc, interpolation="bilinear")
    axes[i].axis("off")
    axes[i].set_title(
        f"{cluster_titles[i]} (Cluster {k})\nN={row['count']:,} ({row['share']}) | Mean Slant: {row['mean_slant']:+.2f}",
        fontsize=13,
        fontweight="bold",
        pad=14
    )

plt.suptitle("Topical Word Clouds Across K-Means Political Clusters (K=3)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 7. Export Clustered Dataset to Parquet
Save the video IDs along with their assigned cluster labels ($K=3$) for downstream analysis.


In [ ]:
CLUSTERS_PARQUET = OUT_DIR / "political_clusters.parquet"
df_clusters_out = pl.from_pandas(df_analysis[["id", "channel", "title", "slant", "political_saliency", "cluster"]])
df_clusters_out.write_parquet(CLUSTERS_PARQUET)
print(f"Successfully saved {len(df_clusters_out):,} clustered political videos to: {CLUSTERS_PARQUET.name}")